# 🛠️ Retail Store Inventory Forecasting — Data Preprocessing

**Objective:** Clean and transform `retail_store_inventory.csv` so that it is ready for model training.

This notebook covers:
1. Loading the Dataset
2. Handling Missing Values
3. Handling Duplicate Records
4. Date Feature Engineering (Year, Month, Day, DayOfWeek, Quarter, IsWeekend)
5. Encoding Categorical Features (Store ID, Product ID, Category, Region, Weather Condition, Seasonality)
6. Final Preprocessed Dataset Overview
7. Saving the Preprocessed Data

---
## 1. Import Libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('retail_store_inventory.csv')
print(f'Dataset loaded successfully.')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

Dataset loaded successfully.
Shape: 73,100 rows × 15 columns


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


---
## 2. Handling Missing Values

In [4]:
# --- 2b. Handle missing values (if any) ---
# Strategy:
#   - Numerical columns → fill with median (robust to outliers)
#   - Categorical columns → fill with mode (most frequent value)

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f'  ✅ Filled "{col}" with median = {median_val}')

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f'  ✅ Filled "{col}" with mode = "{mode_val}"')

if df.isnull().sum().sum() == 0:
    print('\n✅ No missing values found (or all missing values have been handled).')
else:
    print(f'\n⚠️ Remaining missing values: {df.isnull().sum().sum()}')

=== Missing Values Summary ===
                    Missing Count  Missing %
Date                            0        0.0
Store ID                        0        0.0
Product ID                      0        0.0
Category                        0        0.0
Region                          0        0.0
Inventory Level                 0        0.0
Units Sold                      0        0.0
Units Ordered                   0        0.0
Demand Forecast                 0        0.0
Price                           0        0.0
Discount                        0        0.0
Weather Condition               0        0.0
Holiday/Promotion               0        0.0
Competitor Pricing              0        0.0
Seasonality                     0        0.0

Total missing cells : 0
Total cells         : 1,096,500


---
## 3. Handling Duplicate Records

In [5]:
# --- 3a. Check for duplicates ---
dup_count = df.duplicated().sum()
print(f'Duplicate rows found: {dup_count:,}')

if dup_count > 0:
    print(f'  ↳ That is {dup_count / len(df) * 100:.2f}% of the dataset.')
    print('\nSample duplicate rows:')
    display(df[df.duplicated(keep=False)].head(10))

Duplicate rows found: 0


In [6]:
# --- 3b. Remove duplicates ---
rows_before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
rows_after = len(df)

print(f'Rows before removing duplicates : {rows_before:,}')
print(f'Rows after  removing duplicates : {rows_after:,}')
print(f'Rows removed                    : {rows_before - rows_after:,}')
print(f'\n✅ Duplicate handling complete. Current shape: {df.shape}')

Rows before removing duplicates : 73,100
Rows after  removing duplicates : 73,100
Rows removed                    : 0

✅ Duplicate handling complete. Current shape: (73100, 15)


---
## 4. Date Feature Engineering

Convert the `Date` column into the following temporal features:
- **Year** — Calendar year
- **Month** — Month number (1–12)
- **Day** — Day of the month (1–31)
- **DayOfWeek** — Day of the week (0 = Monday … 6 = Sunday)
- **Quarter** — Quarter of the year (1–4)
- **IsWeekend** — Binary flag (1 = Saturday/Sunday, 0 = Weekday)

In [7]:
# --- 4a. Convert Date to datetime ---
df['Date'] = pd.to_datetime(df['Date'])
print(f'Date column dtype: {df["Date"].dtype}')
print(f'Date range: {df["Date"].min()} → {df["Date"].max()}')

Date column dtype: datetime64[ns]
Date range: 2022-01-01 00:00:00 → 2024-01-01 00:00:00


In [8]:
# --- 4b. Extract temporal features ---
df['Year']      = df['Date'].dt.year
df['Month']     = df['Date'].dt.month
df['Day']       = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek          # 0=Mon, 6=Sun
df['Quarter']   = df['Date'].dt.quarter
df['IsWeekend'] = (df['Date'].dt.dayofweek >= 5).astype(int)  # Sat=5, Sun=6

print('✅ Temporal features created successfully.\n')
print(df[['Date', 'Year', 'Month', 'Day', 'DayOfWeek', 'Quarter', 'IsWeekend']].head(10))

✅ Temporal features created successfully.

        Date  Year  Month  Day  DayOfWeek  Quarter  IsWeekend
0 2022-01-01  2022      1    1          5        1          1
1 2022-01-01  2022      1    1          5        1          1
2 2022-01-01  2022      1    1          5        1          1
3 2022-01-01  2022      1    1          5        1          1
4 2022-01-01  2022      1    1          5        1          1
5 2022-01-01  2022      1    1          5        1          1
6 2022-01-01  2022      1    1          5        1          1
7 2022-01-01  2022      1    1          5        1          1
8 2022-01-01  2022      1    1          5        1          1
9 2022-01-01  2022      1    1          5        1          1


In [10]:
# --- 4c. Verify new feature distributions ---
print('=== Temporal Feature Value Counts ===\n')

for col in ['Year', 'Month', 'Quarter', 'DayOfWeek', 'IsWeekend']:
    print(f'--- {col} ---')
    print(df[col].value_counts().sort_index())
    print()

=== Temporal Feature Value Counts ===

--- Year ---
Year
2022    36500
2023    36500
2024      100
Name: count, dtype: int64

--- Month ---
Month
1     6300
2     5600
3     6200
4     6000
5     6200
6     6000
7     6200
8     6200
9     6000
10    6200
11    6000
12    6200
Name: count, dtype: int64

--- Quarter ---
Quarter
1    18100
2    18200
3    18400
4    18400
Name: count, dtype: int64

--- DayOfWeek ---
DayOfWeek
0    10500
1    10400
2    10400
3    10400
4    10400
5    10500
6    10500
Name: count, dtype: int64

--- IsWeekend ---
IsWeekend
0    52100
1    21000
Name: count, dtype: int64



In [11]:
# --- 4d. Drop original Date column (no longer needed) ---
df.drop(columns=['Date'], inplace=True)
print('✅ Dropped original "Date" column.')
print(f'Current shape: {df.shape}')

✅ Dropped original "Date" column.
Current shape: (73100, 20)


---
## 5. Encoding Categorical Features

Apply **Label Encoding** to convert categorical string values into numerical labels:

| Column | Unique Values |
|---|---|
| Store ID | S001 – S005 |
| Product ID | P0001 – P0020 |
| Category | Clothing, Electronics, Furniture, Groceries, Toys |
| Region | East, North, South, West |
| Weather Condition | Cloudy, Rainy, Snowy, Sunny |
| Seasonality | Autumn, Spring, Summer, Winter |

In [12]:
# --- 5a. Inspect unique values before encoding ---
cols_to_encode = ['Store ID', 'Product ID', 'Category', 'Region', 
                  'Weather Condition', 'Seasonality']

print('=== Unique Values (Before Encoding) ===\n')
for col in cols_to_encode:
    unique_vals = sorted(df[col].unique())
    print(f'{col} ({len(unique_vals)} unique): {unique_vals}')

=== Unique Values (Before Encoding) ===

Store ID (5 unique): ['S001', 'S002', 'S003', 'S004', 'S005']
Product ID (20 unique): ['P0001', 'P0002', 'P0003', 'P0004', 'P0005', 'P0006', 'P0007', 'P0008', 'P0009', 'P0010', 'P0011', 'P0012', 'P0013', 'P0014', 'P0015', 'P0016', 'P0017', 'P0018', 'P0019', 'P0020']
Category (5 unique): ['Clothing', 'Electronics', 'Furniture', 'Groceries', 'Toys']
Region (4 unique): ['East', 'North', 'South', 'West']
Weather Condition (4 unique): ['Cloudy', 'Rainy', 'Snowy', 'Sunny']
Seasonality (4 unique): ['Autumn', 'Spring', 'Summer', 'Winter']


In [13]:
# --- 5b. Apply Label Encoding ---
label_encoders = {}   # store encoders for potential inverse transform later

for col in cols_to_encode:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f'✅ Encoded "{col}"  →  Classes: {list(le.classes_)}  →  Codes: {list(range(len(le.classes_)))}')

print(f'\n✅ All {len(cols_to_encode)} categorical columns encoded successfully.')

✅ Encoded "Store ID"  →  Classes: ['S001', 'S002', 'S003', 'S004', 'S005']  →  Codes: [0, 1, 2, 3, 4]
✅ Encoded "Product ID"  →  Classes: ['P0001', 'P0002', 'P0003', 'P0004', 'P0005', 'P0006', 'P0007', 'P0008', 'P0009', 'P0010', 'P0011', 'P0012', 'P0013', 'P0014', 'P0015', 'P0016', 'P0017', 'P0018', 'P0019', 'P0020']  →  Codes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
✅ Encoded "Category"  →  Classes: ['Clothing', 'Electronics', 'Furniture', 'Groceries', 'Toys']  →  Codes: [0, 1, 2, 3, 4]
✅ Encoded "Region"  →  Classes: ['East', 'North', 'South', 'West']  →  Codes: [0, 1, 2, 3]
✅ Encoded "Weather Condition"  →  Classes: ['Cloudy', 'Rainy', 'Snowy', 'Sunny']  →  Codes: [0, 1, 2, 3]
✅ Encoded "Seasonality"  →  Classes: ['Autumn', 'Spring', 'Summer', 'Winter']  →  Codes: [0, 1, 2, 3]

✅ All 6 categorical columns encoded successfully.


In [14]:
# --- 5c. Encoding mapping reference ---
print('=== Encoding Mapping Reference ===\n')

for col, le in label_encoders.items():
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f'{col}:')
    for original, encoded in mapping.items():
        print(f'    {original:20s} → {encoded}')
    print()

=== Encoding Mapping Reference ===

Store ID:
    S001                 → 0
    S002                 → 1
    S003                 → 2
    S004                 → 3
    S005                 → 4

Product ID:
    P0001                → 0
    P0002                → 1
    P0003                → 2
    P0004                → 3
    P0005                → 4
    P0006                → 5
    P0007                → 6
    P0008                → 7
    P0009                → 8
    P0010                → 9
    P0011                → 10
    P0012                → 11
    P0013                → 12
    P0014                → 13
    P0015                → 14
    P0016                → 15
    P0017                → 16
    P0018                → 17
    P0019                → 18
    P0020                → 19

Category:
    Clothing             → 0
    Electronics          → 1
    Furniture            → 2
    Groceries            → 3
    Toys                 → 4

Region:
    East                 → 0
    North   

---
## 6. Final Preprocessed Dataset Overview

In [15]:
# --- 6a. Data types after preprocessing ---
print('=== Final Data Types ===\n')
print(df.dtypes)
print(f'\nShape: {df.shape}')
print(f'Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

=== Final Data Types ===

Store ID                int64
Product ID              int64
Category                int64
Region                  int64
Inventory Level         int64
Units Sold              int64
Units Ordered           int64
Demand Forecast       float64
Price                 float64
Discount                int64
Weather Condition       int64
Holiday/Promotion       int64
Competitor Pricing    float64
Seasonality             int64
Year                    int32
Month                   int32
Day                     int32
DayOfWeek               int32
Quarter                 int32
IsWeekend               int64
dtype: object

Shape: (73100, 20)
Memory Usage: 9.76 MB


In [16]:
# --- 6b. First 10 rows of preprocessed data ---
print('=== Preprocessed Data (First 10 Rows) ===')
df.head(10)

=== Preprocessed Data (First 10 Rows) ===


,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day,DayOfWeek,Quarter,IsWeekend
0,0,0,3,1,231,127,55,135.47,33.50,20,1,0,29.69,0,2022,1,1,5,1,1
1,0,1,4,2,204,150,66,144.04,63.01,20,3,0,66.16,0,2022,1,1,5,1,1
2,0,2,4,3,102,65,51,74.02,27.99,10,3,1,31.32,2,2022,1,1,5,1,1
3,0,3,4,1,469,61,164,62.18,32.72,10,0,1,34.74,0,2022,1,1,5,1,1
4,0,4,1,0,166,14,135,9.26,73.64,0,3,0,68.95,2,2022,1,1,5,1,1
5,0,5,3,2,138,128,102,139.82,76.83,10,3,1,79.35,3,2022,1,1,5,1,1
6,0,6,2,0,359,97,167,108.92,34.16,10,1,1,36.55,3,2022,1,1,5,1,1
7,0,7,0,1,380,312,54,329.73,97.99,5,0,0,100.09,1,2022,1,1,5,1,1
8,0,8,1,3,183,175,135,174.15,20.74,10,0,0,17.66,0,2022,1,1,5,1,1
9,0,9,4,2,108,28,196,24.47,59.99,0,1,1,61.21,3,2022,1,1,5,1,1


In [17]:
# --- 6c. Statistical summary ---
print('=== Statistical Summary ===')
df.describe().T

=== Statistical Summary ===


,count,mean,std,min,25%,50%,75%,max
Store ID,73100.0,2.000000,1.414223,0.00,1.00,2.000,3.0000,4.00
Product ID,73100.0,9.500000,5.766321,0.00,4.75,9.500,14.2500,19.00
Category,73100.0,2.001696,1.414261,0.00,1.00,2.000,3.0000,4.00
Region,73100.0,1.497948,1.118346,0.00,0.00,1.000,2.0000,3.00
Inventory Level,73100.0,274.469877,129.949514,50.00,162.00,273.000,387.0000,500.00
Units Sold,73100.0,136.464870,108.919406,0.00,49.00,107.000,203.0000,499.00
Units Ordered,73100.0,110.004473,52.277448,20.00,65.00,110.000,155.0000,200.00
Demand Forecast,73100.0,141.494720,109.254076,-9.99,53.67,113.015,208.0525,518.55
Price,73100.0,55.135108,26.021945,10.00,32.65,55.050,77.8600,100.00
Discount,73100.0,10.009508,7.083746,0.00,5.00,10.000,15.0000,20.00


In [18]:
# --- 6d. Confirm no remaining issues ---
print('=== Final Sanity Check ===\n')
print(f'Missing values   : {df.isnull().sum().sum()}')
print(f'Duplicate rows   : {df.duplicated().sum()}')
print(f'Object dtypes    : {len(df.select_dtypes(include=["object"]).columns)}')
print(f'Total features   : {df.shape[1]}')
print(f'Total records    : {df.shape[0]:,}')
print(f'\nColumn list: {list(df.columns)}')

=== Final Sanity Check ===

Missing values   : 0
Duplicate rows   : 0
Object dtypes    : 0
Total features   : 20
Total records    : 73,100

Column list: ['Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality', 'Year', 'Month', 'Day', 'DayOfWeek', 'Quarter', 'IsWeekend']


---
## 7. Save Preprocessed Data

In [19]:
output_file = 'preprocessed_retail_inventory.csv'
df.to_csv(output_file, index=False)
print(f'✅ Preprocessed data saved to "{output_file}"')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

✅ Preprocessed data saved to "preprocessed_retail_inventory.csv"
   Shape: 73,100 rows × 20 columns


---
## 📋 Summary

| Step | Action | Details |
|------|--------|---------|
| **Missing Values** | Checked & handled | Numerical → median, Categorical → mode |
| **Duplicates** | Detected & removed | Dropped exact duplicate rows |
| **Date Engineering** | Extracted 6 features | Year, Month, Day, DayOfWeek, Quarter, IsWeekend |
| **Encoding** | Label Encoded 6 columns | Store ID, Product ID, Category, Region, Weather Condition, Seasonality |
| **Output** | Saved preprocessed CSV | `preprocessed_retail_inventory.csv` |